In [21]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm
import pynwb

import bluepyopt as bpop
import bluepyopt.ephys as ephys
from neuron import h
import collections
import json

In [22]:
import efel

# PINN相关导入
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
import torch.nn.functional as F

# 科学计算
from scipy.integrate import odeint
from scipy.optimize import minimize

### Patch data processing

In [23]:
file = '/media/ubuntu/sda/Patch-seq/data/Patch/601790945_icephys.nwb'

io = pynwb.NWBHDF5IO(file, 'r')
data = io.read()

In [37]:
acquisition_dict = {}
count = 0

for i in data.stimulus.keys():
    stimulus = data.get_stimulus(i)
    if stimulus.data_type == 'CurrentClampStimulusSeries':
        acquisition_data = pd.DataFrame(data.get_acquisition(i.split("DA")[0] + "AD0").data, columns = ['acquisition'])
        acquisition_data['stimulus'] = np.array(data.get_stimulus(i).data)
        acquisition_data['time_step'] = range(len(acquisition_data))
        if len(acquisition_data) in [201000, 301000, 401000]:
            acquisition_dict[i] = acquisition_data[45000:115000]
        

In [39]:
def apply_downsampling_to_acquisition_dict(acquisition_dict, downsample_factor=5):
    """
    对acquisition_dict中的所有数据进行降采样
    Args:
        acquisition_dict: 原始数据字典
        downsample_factor: 降采样因子 (5 = 50kHz -> 10kHz)
    Returns:
        降采样后的数据字典
    """
    downsampled_dict = {}
    
    
    for trial_key, trial_data in acquisition_dict.items():
        
        # 降采样
        downsampled_acquisition = trial_data['acquisition'].values[::downsample_factor]
        downsampled_stimulus = trial_data['stimulus'].values[::downsample_factor]
        downsampled_time_step = trial_data['time_step'].values[::downsample_factor]
        
        # 创建新的DataFrame
        downsampled_data = pd.DataFrame({
            'acquisition': downsampled_acquisition,
            'stimulus': downsampled_stimulus,
            'time_step': downsampled_time_step
        })
        
        downsampled_dict[trial_key] = downsampled_data
            
    return downsampled_dict


acquisition_dict_10khz = apply_downsampling_to_acquisition_dict(acquisition_dict, downsample_factor=5)

In [33]:
with PdfPages("Current_clamp_results.pdf") as pdf:
    for key, item in acquisition_dict.items():
        plt.figure(figsize=(6, 3))
        sns.lineplot(x = item['time_step'],
                     y = item['stimulus'])
        sns.lineplot(x = item['time_step'],
                     y = item['acquisition'])
        pdf.savefig()
        plt.close()

In [ ]:
# RNN-based PINN for HH Model
# 结合循环神经网络和物理约束的HH模型拟合

class RNN_HH_PINN(nn.Module):
    """
    基于RNN的PINN网络，用于HH模型时序拟合
    核心思想：V(t), m(t), h(t), n(t) = f(V(t-1), m(t-1), h(t-1), n(t-1), I(t))
    """
    def __init__(self, hidden_size=128, num_layers=2, dropout=0.1):
        super(RNN_HH_PINN, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # RNN层：处理时序信息
        # 输入：[V(t-1), m(t-1), h(t-1), n(t-1), I(t)] -> 5维
        # 输出：隐藏状态
        self.rnn = nn.LSTM(
            input_size=5,  # V, m, h, n, I
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # 输出层：从隐藏状态到HH变量
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 4)  # V, m, h, n
        )
        
        # HH模型参数（可学习）
        self.g_Na = nn.Parameter(torch.tensor(120.0))
        self.g_K = nn.Parameter(torch.tensor(36.0))
        self.g_L = nn.Parameter(torch.tensor(0.3))
        self.E_Na = nn.Parameter(torch.tensor(50.0))
        self.E_K = nn.Parameter(torch.tensor(-77.0))
        self.E_L = nn.Parameter(torch.tensor(-54.4))
        self.C_m = nn.Parameter(torch.tensor(1.0))
        
        # 时间步长（可学习）
        self.dt = nn.Parameter(torch.tensor(0.01))  # 10ms
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重"""
        for name, param in self.rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
        
        for m in self.output_layer.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, sequence_input, hidden=None):
        """
        前向传播
        Args:
            sequence_input: [batch_size, seq_len, 5] -> [V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            hidden: 初始隐藏状态
        Returns:
            outputs: [batch_size, seq_len, 4] -> [V(t), m(t), h(t), n(t)]
            hidden: 最终隐藏状态
        """
        batch_size, seq_len, _ = sequence_input.shape
        
        # RNN处理
        rnn_output, hidden = self.rnn(sequence_input, hidden)
        
        # 输出层
        raw_outputs = self.output_layer(rnn_output)
        
        # 分离并约束输出
        V = raw_outputs[:, :, 0:1]
        m = torch.sigmoid(raw_outputs[:, :, 1:2])  # [0,1]
        h = torch.sigmoid(raw_outputs[:, :, 2:3])  # [0,1]
        n = torch.sigmoid(raw_outputs[:, :, 3:4])  # [0,1]
        
        return torch.cat([V, m, h, n], dim=-1), hidden
    
    def init_hidden(self, batch_size, device):
        """初始化隐藏状态"""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        return (h0, c0)
    
    def predict_sequence(self, initial_state, current_sequence, hidden=None):
        """
        预测整个序列
        Args:
            initial_state: [batch_size, 4] -> [V(0), m(0), h(0), n(0)]
            current_sequence: [batch_size, seq_len, 1] -> I(t)
            hidden: 初始隐藏状态
        Returns:
            predicted_sequence: [batch_size, seq_len, 4]
        """
        batch_size, seq_len, _ = current_sequence.shape
        device = initial_state.device
        
        if hidden is None:
            hidden = self.init_hidden(batch_size, device)
        
        # 存储预测结果
        predictions = []
        current_state = initial_state
        
        for t in range(seq_len):
            # 构建输入：[V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            current_input = torch.cat([current_state, current_sequence[:, t:t+1, :]], dim=-1)
            current_input = current_input.unsqueeze(1)  # [batch_size, 1, 5]
            
            # 前向传播
            output, hidden = self.forward(current_input, hidden)
            current_state = output.squeeze(1)  # [batch_size, 4]
            predictions.append(current_state)
        
        return torch.stack(predictions, dim=1)  # [batch_size, seq_len, 4]


In [ ]:
class RNN_HH_PhysicsConstraints:
    """
    基于RNN的HH模型时序物理约束
    考虑时序依赖性的物理约束
    """
    
    def __init__(self, model):
        self.model = model
    
    def alpha_m(self, V):
        """钠通道m门控变量的激活速率"""
        return 0.1 * (V + 40.0) / (1.0 - torch.exp(-(V + 40.0) / 10.0))
    
    def beta_m(self, V):
        """钠通道m门控变量的失活速率"""
        return 4.0 * torch.exp(-(V + 65.0) / 18.0)
    
    def alpha_h(self, V):
        """钠通道h门控变量的激活速率"""
        return 0.07 * torch.exp(-(V + 65.0) / 20.0)
    
    def beta_h(self, V):
        """钠通道h门控变量的失活速率"""
        return 1.0 / (1.0 + torch.exp(-(V + 35.0) / 10.0))
    
    def alpha_n(self, V):
        """钾通道n门控变量的激活速率"""
        return 0.01 * (V + 55.0) / (1.0 - torch.exp(-(V + 55.0) / 10.0))
    
    def beta_n(self, V):
        """钾通道n门控变量的失活速率"""
        return 0.125 * torch.exp(-(V + 65.0) / 80.0)
    
    def compute_temporal_physics_loss(self, sequence_input, predicted_outputs):
        """
        计算时序物理约束损失
        Args:
            sequence_input: [batch_size, seq_len, 5] -> [V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            predicted_outputs: [batch_size, seq_len, 4] -> [V(t), m(t), h(t), n(t)]
        """
        batch_size, seq_len, _ = predicted_outputs.shape
        
        # 分离输入和输出
        V_prev = sequence_input[:, :, 0:1]  # V(t-1)
        m_prev = sequence_input[:, :, 1:2]  # m(t-1)
        h_prev = sequence_input[:, :, 2:3]  # h(t-1)
        n_prev = sequence_input[:, :, 3:4]  # n(t-1)
        I = sequence_input[:, :, 4:5]       # I(t)
        
        V_curr = predicted_outputs[:, :, 0:1]  # V(t)
        m_curr = predicted_outputs[:, :, 1:2]  # m(t)
        h_curr = predicted_outputs[:, :, 2:3]  # h(t)
        n_curr = predicted_outputs[:, :, 3:4]  # n(t)
        
        # 计算时间导数（有限差分）
        dt = self.model.dt
        dV_dt = (V_curr - V_prev) / dt
        dm_dt = (m_curr - m_prev) / dt
        dh_dt = (h_curr - h_prev) / dt
        dn_dt = (n_curr - n_prev) / dt
        
        # HH方程约束
        # 电压方程：C_m * dV/dt = I - g_Na*m^3*h*(V-E_Na) - g_K*n^4*(V-E_K) - g_L*(V-E_L)
        I_Na = self.model.g_Na * (m_curr**3) * h_curr * (V_curr - self.model.E_Na)
        I_K = self.model.g_K * (n_curr**4) * (V_curr - self.model.E_K)
        I_L = self.model.g_L * (V_curr - self.model.E_L)
        
        voltage_eq = self.model.C_m * dV_dt - (I - I_Na - I_K - I_L)
        
        # 门控变量方程
        m_eq = dm_dt - (self.alpha_m(V_prev) * (1 - m_prev) - self.beta_m(V_prev) * m_prev)
        h_eq = dh_dt - (self.alpha_h(V_prev) * (1 - h_prev) - self.beta_h(V_prev) * h_prev)
        n_eq = dn_dt - (self.alpha_n(V_prev) * (1 - n_prev) - self.beta_n(V_prev) * n_prev)
        
        # 物理约束损失
        physics_loss = torch.mean(voltage_eq**2) + torch.mean(m_eq**2) + \
                      torch.mean(h_eq**2) + torch.mean(n_eq**2)
        
        return physics_loss
    
    def compute_continuity_loss(self, predicted_outputs):
        """
        计算连续性损失
        确保相邻时间步之间的变化是平滑的
        """
        if predicted_outputs.shape[1] < 2:
            return torch.tensor(0.0, device=predicted_outputs.device)
        
        # 计算相邻时间步的差异
        diff = predicted_outputs[:, 1:, :] - predicted_outputs[:, :-1, :]
        
        # 连续性损失：相邻时间步的变化应该相对较小
        continuity_loss = torch.mean(diff**2)
        
        return continuity_loss
    
    def compute_initial_conditions_loss(self, initial_state):
        """
        计算初始条件损失
        Args:
            initial_state: [batch_size, 4] -> [V(0), m(0), h(0), n(0)]
        """
        V_init = initial_state[:, 0:1]
        m_init = initial_state[:, 1:2]
        h_init = initial_state[:, 2:3]
        n_init = initial_state[:, 3:4]
        
        # 初始条件约束
        V_init_loss = torch.mean((V_init + 65.0)**2)  # 静息电位约-65mV
        m_init_loss = torch.mean((m_init - 0.05)**2)  # m初始值约0.05
        h_init_loss = torch.mean((h_init - 0.6)**2)   # h初始值约0.6
        n_init_loss = torch.mean((n_init - 0.32)**2)  # n初始值约0.32
        
        return V_init_loss + m_init_loss + h_init_loss + n_init_loss
    
    def compute_biological_constraints_loss(self, predicted_outputs):
        """
        计算生物学约束损失
        确保输出符合生物学合理性
        """
        V = predicted_outputs[:, :, 0:1]
        m = predicted_outputs[:, :, 1:2]
        h = predicted_outputs[:, :, 2:3]
        n = predicted_outputs[:, :, 3:4]
        
        # 电压范围约束：-100mV 到 50mV
        V_min_loss = torch.mean(torch.relu(-100.0 - V)**2)
        V_max_loss = torch.mean(torch.relu(V - 50.0)**2)
        
        # 门控变量范围约束：[0,1]（通过sigmoid已经保证，但可以添加额外约束）
        m_range_loss = torch.mean(torch.relu(m - 1.0)**2) + torch.mean(torch.relu(-m)**2)
        h_range_loss = torch.mean(torch.relu(h - 1.0)**2) + torch.mean(torch.relu(-h)**2)
        n_range_loss = torch.mean(torch.relu(n - 1.0)**2) + torch.mean(torch.relu(-n)**2)
        
        return V_min_loss + V_max_loss + m_range_loss + h_range_loss + n_range_loss


In [ ]:
# 更新seq_length为200 (20ms)
# 修改RNN_PatchDataProcessor的默认序列长度

# 重新定义RNN_PatchDataProcessor类，使用seq_length=200
class RNN_PatchDataProcessor:
    """
    基于RNN的电流钳数据处理器
    将数据转换为序列格式，适应RNN训练
    使用10kHz降采样数据，seq_length=200 (20ms)
    """
    
    def __init__(self, acquisition_dict, sequence_length=200):
        self.acquisition_dict = acquisition_dict
        self.processed_data = {}
        self.sequence_length = sequence_length
        self.sampling_rate = 10000  # 10kHz降采样后的采样率
        
        print(f"初始化RNN数据处理器:")
        print(f"  序列长度: {sequence_length} 个采样点 ({sequence_length/self.sampling_rate*1000:.1f}ms)")
        print(f"  采样率: {self.sampling_rate}Hz")
        print(f"  时间分辨率: {1/self.sampling_rate*1000:.1f}ms")
    
    def process_single_trial(self, trial_key, trial_data):
        """
        处理单个试验的数据
        """
        # 提取时间和电压数据
        time_points = trial_data['time_step'].values / self.sampling_rate  # 转换为秒
        voltage = trial_data['acquisition'].values
        current = trial_data['stimulus'].values
        
        # 数据预处理
        voltage_baseline = np.mean(voltage[:200])  # 前20ms作为基线 (10kHz采样率)
        voltage_corrected = voltage - voltage_baseline
        
        # 平滑处理
        from scipy.signal import savgol_filter
        voltage_smooth = savgol_filter(voltage_corrected, window_length=21, polyorder=3)
        
        # 归一化
        voltage_norm = voltage_smooth / 100.0
        current_norm = current / 1000.0
        
        return {
            'time': time_points,
            'voltage': voltage_norm,
            'current': current_norm,
            'voltage_raw': voltage,
            'current_raw': current
        }
    
    def process_all_trials(self):
        """处理所有试验数据"""
        for trial_key, trial_data in self.acquisition_dict.items():
            print(f"处理试验: {trial_key}")
            processed = self.process_single_trial(trial_key, trial_data)
            self.processed_data[trial_key] = processed
    
    def create_sequences(self, data, sequence_length=None):
        """
        创建序列数据
        Args:
            data: 包含time, voltage, current的字典
            sequence_length: 序列长度
        """
        if sequence_length is None:
            sequence_length = self.sequence_length
        
        times = data['time']
        voltages = data['voltage']
        currents = data['current']
        
        sequences = []
        targets = []
        initial_states = []
        
        # 滑动窗口创建序列，步长为序列长度的一半（50%重叠）
        step_size = sequence_length // 2
        
        for i in range(0, len(times) - sequence_length, step_size):
            # 输入序列：[V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            seq_input = []
            seq_target = []
            
            for j in range(sequence_length):
                t_idx = i + j
                if t_idx < len(times) - 1:
                    # 构建输入：[V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
                    V_prev = voltages[t_idx]
                    m_prev = 0.5  # 简化的初始值
                    h_prev = 0.5
                    n_prev = 0.5
                    I_curr = currents[t_idx + 1]
                    
                    seq_input.append([V_prev, m_prev, h_prev, n_prev, I_curr])
                    
                    # 目标：[V(t), m(t), h(t), n(t)]
                    V_curr = voltages[t_idx + 1]
                    seq_target.append([V_curr, m_prev, h_prev, n_prev])  # 简化的目标
            
            if len(seq_input) == sequence_length:
                sequences.append(seq_input)
                targets.append(seq_target)
                
                # 初始状态
                initial_state = [voltages[i], 0.5, 0.5, 0.5]  # V(0), m(0), h(0), n(0)
                initial_states.append(initial_state)
        
        return np.array(sequences), np.array(targets), np.array(initial_states)
    
    def create_training_dataset(self, train_ratio=0.8):
        """
        创建训练数据集
        """
        all_sequences = []
        all_targets = []
        all_initial_states = []
        
        for trial_data in self.processed_data.values():
            sequences, targets, initial_states = self.create_sequences(trial_data)
            all_sequences.extend(sequences)
            all_targets.extend(targets)
            all_initial_states.extend(initial_states)
        
        # 转换为numpy数组
        sequences = np.array(all_sequences)
        targets = np.array(all_targets)
        initial_states = np.array(all_initial_states)
        
        print(f"\\n序列创建完成:")
        print(f"  总序列数: {len(sequences)}")
        print(f"  序列形状: {sequences.shape}")
        print(f"  每个序列长度: {self.sequence_length} 个采样点 ({self.sequence_length/self.sampling_rate*1000:.1f}ms)")
        print(f"  序列重叠: 50% (步长: {self.sequence_length//2} 个采样点)")
        
        # 分割训练和测试集
        n_samples = len(sequences)
        n_train = int(n_samples * train_ratio)
        
        indices = np.random.permutation(n_samples)
        train_indices = indices[:n_train]
        test_indices = indices[n_train:]
        
        train_data = {
            'sequences': sequences[train_indices],
            'targets': targets[train_indices],
            'initial_states': initial_states[train_indices]
        }
        
        test_data = {
            'sequences': sequences[test_indices],
            'targets': targets[test_indices],
            'initial_states': initial_states[test_indices]
        }
        
        print(f"  训练序列数: {len(train_data['sequences'])}")
        print(f"  测试序列数: {len(test_data['sequences'])}")
        
        return train_data, test_data
    
    def plot_sample_sequences(self, trial_key=None, n_sequences=3):
        """
        绘制样本序列
        """
        if trial_key is None:
            trial_key = list(self.processed_data.keys())[0]
        
        data = self.processed_data[trial_key]
        sequences, targets, initial_states = self.create_sequences(data)
        
        plt.figure(figsize=(15, 10))
        
        for i in range(min(n_sequences, len(sequences))):
            plt.subplot(n_sequences, 1, i + 1)
            
            # 输入序列
            seq_input = sequences[i]
            V_input = seq_input[:, 0]
            I_input = seq_input[:, 4]
            
            # 目标序列
            seq_target = targets[i]
            V_target = seq_target[:, 0]
            
            # 绘制
            time_steps = range(len(V_input))
            plt.plot(time_steps, V_input, 'b-', label='输入电压', alpha=0.7)
            plt.plot(time_steps, V_target, 'r-', label='目标电压', alpha=0.7)
            plt.plot(time_steps, I_input, 'g-', label='电流', alpha=0.5)
            
            plt.title(f'序列 {i+1} (长度: {self.sequence_length} 采样点, {self.sequence_length/self.sampling_rate*1000:.1f}ms)')
            plt.xlabel('时间步')
            plt.ylabel('归一化值')
            plt.legend()
            plt.grid(True)
        
        plt.tight_layout()
        plt.show()

# RNN数据集类
class RNN_HH_Dataset(Dataset):
    """
    RNN HH模型数据集
    """
    def __init__(self, data_dict):
        self.sequences = torch.FloatTensor(data_dict['sequences'])
        self.targets = torch.FloatTensor(data_dict['targets'])
        self.initial_states = torch.FloatTensor(data_dict['initial_states'])
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx], self.initial_states[idx]

def create_rnn_data_loaders(train_data, test_data, batch_size=32):
    """
    创建RNN数据加载器
    """
    train_dataset = RNN_HH_Dataset(train_data)
    test_dataset = RNN_HH_Dataset(test_data)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

print("RNN_PatchDataProcessor已更新为seq_length=200 (20ms)")


In [ ]:
# 修改损失函数：只使用物理约束，去掉数据约束

class RNN_PINN_LossFunction:
    """
    基于RNN的PINN损失函数
    只使用物理约束，不使用数据约束
    """
    
    def __init__(self, physics_constraints, 
                 lambda_physics=1.0, 
                 lambda_init=0.1,
                 lambda_continuity=0.05,
                 lambda_biological=0.01):
        self.physics_constraints = physics_constraints
        self.lambda_physics = lambda_physics     # 物理约束损失权重
        self.lambda_init = lambda_init           # 初始条件损失权重
        self.lambda_continuity = lambda_continuity  # 连续性损失权重
        self.lambda_biological = lambda_biological  # 生物学约束损失权重
    
    def compute_total_loss(self, sequence_input, predicted_outputs, true_outputs, initial_state):
        """
        计算总损失函数 - 只使用物理约束
        Args:
            sequence_input: [batch_size, seq_len, 5] -> [V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            predicted_outputs: [batch_size, seq_len, 4] -> [V(t), m(t), h(t), n(t)]
            true_outputs: [batch_size, seq_len, 4] -> 真实值 (仅用于参考，不参与损失计算)
            initial_state: [batch_size, 4] -> 初始状态
        """
        # 时序物理约束损失
        physics_loss = self.physics_constraints.compute_temporal_physics_loss(
            sequence_input, predicted_outputs
        )
        
        # 初始条件损失
        init_loss = self.physics_constraints.compute_initial_conditions_loss(initial_state)
        
        # 连续性损失
        continuity_loss = self.physics_constraints.compute_continuity_loss(predicted_outputs)
        
        # 生物学约束损失
        biological_loss = self.physics_constraints.compute_biological_constraints_loss(predicted_outputs)
        
        # 总损失 - 只使用物理约束
        total_loss = (self.lambda_physics * physics_loss + 
                     self.lambda_init * init_loss +
                     self.lambda_continuity * continuity_loss +
                     self.lambda_biological * biological_loss)
        
        return total_loss, physics_loss, init_loss, continuity_loss, biological_loss

class RNN_PINN_Trainer:
    """
    基于RNN的PINN训练器 - 只使用物理约束
    """
    
    def __init__(self, model, loss_function, learning_rate=1e-3):
        self.model = model
        self.loss_function = loss_function
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=50, verbose=True
        )
        
        # 训练历史
        self.train_losses = []
        self.physics_losses = []
        self.init_losses = []
        self.continuity_losses = []
        self.biological_losses = []
    
    def train_step(self, sequence_input, true_outputs, initial_state):
        """
        单步训练
        Args:
            sequence_input: [batch_size, seq_len, 5]
            true_outputs: [batch_size, seq_len, 4] (仅用于参考)
            initial_state: [batch_size, 4]
        """
        self.optimizer.zero_grad()
        
        # 前向传播
        predicted_outputs, hidden = self.model(sequence_input)
        
        # 计算损失 - 只使用物理约束
        total_loss, physics_loss, init_loss, continuity_loss, biological_loss = \
            self.loss_function.compute_total_loss(sequence_input, predicted_outputs, true_outputs, initial_state)
        
        # 反向传播
        total_loss.backward()
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        
        self.optimizer.step()
        
        return (total_loss.item(), physics_loss.item(), 
                init_loss.item(), continuity_loss.item(), biological_loss.item())
    
    def train(self, train_loader, num_epochs=1000, print_interval=100):
        """
        训练RNN PINN模型 - 只使用物理约束
        """
        self.model.train()
        
        for epoch in range(num_epochs):
            epoch_losses = []
            epoch_physics_losses = []
            epoch_init_losses = []
            epoch_continuity_losses = []
            epoch_biological_losses = []
            
            for batch_idx, (sequence_input, true_outputs, initial_state) in enumerate(train_loader):
                losses = self.train_step(sequence_input, true_outputs, initial_state)
                
                total_loss, physics_loss, init_loss, continuity_loss, biological_loss = losses
                
                epoch_losses.append(total_loss)
                epoch_physics_losses.append(physics_loss)
                epoch_init_losses.append(init_loss)
                epoch_continuity_losses.append(continuity_loss)
                epoch_biological_losses.append(biological_loss)
            
            # 记录平均损失
            avg_total_loss = np.mean(epoch_losses)
            avg_physics_loss = np.mean(epoch_physics_losses)
            avg_init_loss = np.mean(epoch_init_losses)
            avg_continuity_loss = np.mean(epoch_continuity_losses)
            avg_biological_loss = np.mean(epoch_biological_losses)
            
            self.train_losses.append(avg_total_loss)
            self.physics_losses.append(avg_physics_loss)
            self.init_losses.append(avg_init_loss)
            self.continuity_losses.append(avg_continuity_loss)
            self.biological_losses.append(avg_biological_loss)
            
            # 学习率调度
            self.scheduler.step(avg_total_loss)
            
            # 打印训练进度
            if epoch % print_interval == 0:
                print(f'Epoch {epoch:4d}: Total={avg_total_loss:.6f}, '
                      f'Physics={avg_physics_loss:.6f}, '
                      f'Init={avg_init_loss:.6f}, Cont={avg_continuity_loss:.6f}, '
                      f'Bio={avg_biological_loss:.6f}')
    
    def plot_training_history(self):
        """绘制训练历史"""
        plt.figure(figsize=(15, 10))
        
        plt.subplot(2, 3, 1)
        plt.plot(self.train_losses)
        plt.title('Total Loss (Physics Only)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 2)
        plt.plot(self.physics_losses)
        plt.title('Physics Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 3)
        plt.plot(self.init_losses)
        plt.title('Initial Conditions Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 4)
        plt.plot(self.continuity_losses)
        plt.title('Continuity Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 5)
        plt.plot(self.biological_losses)
        plt.title('Biological Constraints Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.tight_layout()
        plt.show()

# 更新训练调用，使用纯物理约束
if 'acquisition_dict_10khz' in locals() and len(acquisition_dict_10khz) > 0:
    print("发现10kHz降采样电流钳数据，开始RNN-based PINN训练...")
    print("使用纯物理约束损失函数 (无数据约束)")
    print("seq_length=200 (20ms)，适合RNN上下文长度")
    
    # 训练模型
    rnn_model, rnn_trainer, rnn_processor = train_rnn_hh_pinn(
        acquisition_dict_10khz, 
        num_epochs=500,  # 可以根据需要调整
        batch_size=16,   # RNN通常需要较小的batch size
        learning_rate=1e-3,
        sequence_length=200  # 20ms序列长度
    )
    
    # 分析学习到的参数
    analyze_rnn_learned_parameters(rnn_model)
    
else:
    print("未发现10kHz降采样数据，请先运行降采样处理部分...")


In [ ]:
# 更新训练函数，使用纯物理约束损失函数

def train_rnn_hh_pinn(acquisition_dict_10khz, num_epochs=1000, batch_size=32, learning_rate=1e-3, sequence_length=200):
    """
    训练基于RNN的HH PINN模型 (使用10kHz降采样数据，纯物理约束)
    """
    print("开始RNN-based PINN训练 (使用10kHz降采样数据，纯物理约束)...")
    
    # 1. 数据处理
    print("1. 处理10kHz降采样电流钳数据为序列格式...")
    processor = RNN_PatchDataProcessor(acquisition_dict_10khz, sequence_length=sequence_length)
    processor.process_all_trials()
    
    # 可视化样本序列
    processor.plot_sample_sequences()
    
    # 创建训练数据集
    train_data, test_data = processor.create_training_dataset()
    train_loader, test_loader = create_rnn_data_loaders(train_data, test_data, batch_size)
    
    print(f"训练序列数: {len(train_data['sequences'])}")
    print(f"测试序列数: {len(test_data['sequences'])}")
    
    # 2. 初始化模型
    print("2. 初始化RNN PINN模型...")
    model = RNN_HH_PINN(hidden_size=128, num_layers=2, dropout=0.1)
    physics_constraints = RNN_HH_PhysicsConstraints(model)
    
    # 使用纯物理约束损失函数
    loss_function = RNN_PINN_LossFunction(physics_constraints, 
                                        lambda_physics=1.0, 
                                        lambda_init=0.1,
                                        lambda_continuity=0.05,
                                        lambda_biological=0.01)
    trainer = RNN_PINN_Trainer(model, loss_function, learning_rate)
    
    # 打印模型参数
    print(f"模型参数数量: {sum(p.numel() for p in model.parameters())}")
    print(f"HH模型参数:")
    print(f"  g_Na: {model.g_Na.item():.2f}")
    print(f"  g_K: {model.g_K.item():.2f}")
    print(f"  g_L: {model.g_L.item():.2f}")
    print(f"  E_Na: {model.E_Na.item():.2f}")
    print(f"  E_K: {model.E_K.item():.2f}")
    print(f"  E_L: {model.E_L.item():.2f}")
    print(f"  C_m: {model.C_m.item():.2f}")
    print(f"  dt: {model.dt.item():.4f}")
    
    # 3. 训练模型
    print("3. 开始训练 (纯物理约束)...")
    trainer.train(train_loader, num_epochs=num_epochs, print_interval=100)
    
    # 4. 训练历史可视化
    print("4. 绘制训练历史...")
    trainer.plot_training_history()
    
    # 5. 模型评估
    print("5. 评估模型性能...")
    evaluate_rnn_model(model, test_loader, processor)
    
    return model, trainer, processor

def evaluate_rnn_model(model, test_loader, processor):
    """
    评估RNN模型性能 (纯物理约束)
    """
    model.eval()
    
    with torch.no_grad():
        test_losses = []
        all_predictions = []
        all_targets = []
        
        for sequence_input, true_outputs, initial_state in test_loader:
            predicted_outputs, hidden = model(sequence_input)
            
            # 计算测试损失 - 只使用物理约束
            physics_constraints = RNN_HH_PhysicsConstraints(model)
            loss_function = RNN_PINN_LossFunction(physics_constraints)
            total_loss, physics_loss, init_loss, continuity_loss, biological_loss = \
                loss_function.compute_total_loss(sequence_input, predicted_outputs, true_outputs, initial_state)
            
            test_losses.append(total_loss.item())
            
            # 收集预测结果
            all_predictions.extend(predicted_outputs.cpu().numpy())
            all_targets.extend(true_outputs.cpu().numpy())
        
        avg_test_loss = np.mean(test_losses)
        print(f"平均测试损失 (纯物理约束): {avg_test_loss:.6f}")
        
        # 计算相关系数
        predictions = np.array(all_predictions)
        targets = np.array(all_targets)
        
        # 计算电压预测的相关系数
        V_pred = predictions[:, :, 0].flatten()
        V_true = targets[:, :, 0].flatten()
        correlation = np.corrcoef(V_pred, V_true)[0, 1]
        print(f"电压预测与真实值的相关系数: {correlation:.4f}")
        
        # 绘制预测结果
        plot_rnn_predictions(predictions, targets, processor)

def plot_rnn_predictions(predictions, targets, processor, n_samples=5):
    """
    绘制RNN预测结果
    """
    plt.figure(figsize=(15, 12))
    
    # 随机选择样本进行可视化
    n_total = len(predictions)
    indices = np.random.choice(n_total, min(n_samples, n_total), replace=False)
    
    for i, idx in enumerate(indices):
        plt.subplot(n_samples, 1, i + 1)
        
        pred_sample = predictions[idx]
        target_sample = targets[idx]
        
        V_pred = pred_sample[:, 0]
        V_true = target_sample[:, 0]
        
        time_steps = range(len(V_pred))
        plt.plot(time_steps, V_true, 'b-', label='真实电压', alpha=0.8)
        plt.plot(time_steps, V_pred, 'r--', label='预测电压', alpha=0.8)
        
        plt.title(f'序列 {i+1} - 电压预测 (纯物理约束)')
        plt.xlabel('时间步')
        plt.ylabel('归一化电压')
        plt.legend()
        plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # 预测vs真实值散点图
    plt.figure(figsize=(10, 8))
    
    V_pred_all = predictions[:, :, 0].flatten()
    V_true_all = targets[:, :, 0].flatten()
    
    # 随机采样用于散点图
    n_scatter = min(5000, len(V_pred_all))
    scatter_indices = np.random.choice(len(V_pred_all), n_scatter, replace=False)
    
    plt.scatter(V_true_all[scatter_indices], V_pred_all[scatter_indices], 
                alpha=0.5, s=1, c='blue')
    plt.plot([V_true_all.min(), V_true_all.max()], 
             [V_true_all.min(), V_true_all.max()], 'r--', lw=2)
    plt.xlabel('真实电压')
    plt.ylabel('预测电压')
    plt.title('电压预测vs真实值 (纯物理约束)')
    plt.grid(True)
    plt.show()

def analyze_rnn_learned_parameters(model):
    """
    分析RNN学习到的HH模型参数
    """
    print("RNN学习到的HH模型参数 (纯物理约束训练):")
    print(f"钠电导 (g_Na): {model.g_Na.item():.2f} mS/cm²")
    print(f"钾电导 (g_K): {model.g_K.item():.2f} mS/cm²")
    print(f"漏电导 (g_L): {model.g_L.item():.2f} mS/cm²")
    print(f"钠平衡电位 (E_Na): {model.E_Na.item():.2f} mV")
    print(f"钾平衡电位 (E_K): {model.E_K.item():.2f} mV")
    print(f"漏平衡电位 (E_L): {model.E_L.item():.2f} mV")
    print(f"膜电容 (C_m): {model.C_m.item():.2f} μF/cm²")
    print(f"时间步长 (dt): {model.dt.item():.4f} s")
    
    # 与标准HH参数比较
    standard_params = {
        'g_Na': 120.0, 'g_K': 36.0, 'g_L': 0.3,
        'E_Na': 50.0, 'E_K': -77.0, 'E_L': -54.4, 'C_m': 1.0
    }
    
    print("\n与标准HH参数比较:")
    for param_name, standard_value in standard_params.items():
        learned_value = getattr(model, param_name).item()
        diff = abs(learned_value - standard_value)
        diff_pct = (diff / standard_value) * 100
        print(f"{param_name}: 学习值={learned_value:.2f}, 标准值={standard_value:.2f}, "
              f"差异={diff:.2f} ({diff_pct:.1f}%)")

print("训练函数已更新为使用纯物理约束损失函数")


In [ ]:
# 重新定义损失函数：只使用HH微分方程约束，为四个方程分别设置权重

class RNN_PINN_LossFunction:
    """
    基于RNN的PINN损失函数
    只使用HH微分方程约束，为四个方程分别设置权重
    """
    
    def __init__(self, physics_constraints, 
                 lambda_voltage=1.0, 
                 lambda_m=1.0,
                 lambda_h=1.0,
                 lambda_n=1.0):
        self.physics_constraints = physics_constraints
        self.lambda_voltage = lambda_voltage  # 电压方程权重
        self.lambda_m = lambda_m             # m门控变量方程权重
        self.lambda_h = lambda_h             # h门控变量方程权重
        self.lambda_n = lambda_n             # n门控变量方程权重
    
    def compute_total_loss(self, sequence_input, predicted_outputs, true_outputs, initial_state):
        """
        计算总损失函数 - 只使用HH微分方程约束
        Args:
            sequence_input: [batch_size, seq_len, 5] -> [V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            predicted_outputs: [batch_size, seq_len, 4] -> [V(t), m(t), h(t), n(t)]
            true_outputs: [batch_size, seq_len, 4] -> 真实值 (仅用于参考，不参与损失计算)
            initial_state: [batch_size, 4] -> 初始状态 (仅用于参考，不参与损失计算)
        """
        # 计算HH微分方程约束损失
        voltage_loss, m_loss, h_loss, n_loss = self.physics_constraints.compute_hh_equations_loss(
            sequence_input, predicted_outputs
        )
        
        # 总损失 - 四个方程分别加权
        total_loss = (self.lambda_voltage * voltage_loss + 
                     self.lambda_m * m_loss +
                     self.lambda_h * h_loss +
                     self.lambda_n * n_loss)
        
        return total_loss, voltage_loss, m_loss, h_loss, n_loss

class RNN_PINN_Trainer:
    """
    基于RNN的PINN训练器 - 只使用HH微分方程约束
    """
    
    def __init__(self, model, loss_function, learning_rate=1e-3):
        self.model = model
        self.loss_function = loss_function
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=50, verbose=True
        )
        
        # 训练历史
        self.train_losses = []
        self.voltage_losses = []
        self.m_losses = []
        self.h_losses = []
        self.n_losses = []
    
    def train_step(self, sequence_input, true_outputs, initial_state):
        """
        单步训练
        Args:
            sequence_input: [batch_size, seq_len, 5]
            true_outputs: [batch_size, seq_len, 4] (仅用于参考)
            initial_state: [batch_size, 4] (仅用于参考)
        """
        self.optimizer.zero_grad()
        
        # 前向传播
        predicted_outputs, hidden = self.model(sequence_input)
        
        # 计算损失 - 只使用HH微分方程约束
        total_loss, voltage_loss, m_loss, h_loss, n_loss = \
            self.loss_function.compute_total_loss(sequence_input, predicted_outputs, true_outputs, initial_state)
        
        # 反向传播
        total_loss.backward()
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        
        self.optimizer.step()
        
        return (total_loss.item(), voltage_loss.item(), 
                m_loss.item(), h_loss.item(), n_loss.item())
    
    def train(self, train_loader, num_epochs=1000, print_interval=100):
        """
        训练RNN PINN模型 - 只使用HH微分方程约束
        """
        self.model.train()
        
        for epoch in range(num_epochs):
            epoch_losses = []
            epoch_voltage_losses = []
            epoch_m_losses = []
            epoch_h_losses = []
            epoch_n_losses = []
            
            for batch_idx, (sequence_input, true_outputs, initial_state) in enumerate(train_loader):
                losses = self.train_step(sequence_input, true_outputs, initial_state)
                
                total_loss, voltage_loss, m_loss, h_loss, n_loss = losses
                
                epoch_losses.append(total_loss)
                epoch_voltage_losses.append(voltage_loss)
                epoch_m_losses.append(m_loss)
                epoch_h_losses.append(h_loss)
                epoch_n_losses.append(n_loss)
            
            # 记录平均损失
            avg_total_loss = np.mean(epoch_losses)
            avg_voltage_loss = np.mean(epoch_voltage_losses)
            avg_m_loss = np.mean(epoch_m_losses)
            avg_h_loss = np.mean(epoch_h_losses)
            avg_n_loss = np.mean(epoch_n_losses)
            
            self.train_losses.append(avg_total_loss)
            self.voltage_losses.append(avg_voltage_loss)
            self.m_losses.append(avg_m_loss)
            self.h_losses.append(avg_h_loss)
            self.n_losses.append(avg_n_loss)
            
            # 学习率调度
            self.scheduler.step(avg_total_loss)
            
            # 打印训练进度
            if epoch % print_interval == 0:
                print(f'Epoch {epoch:4d}: Total={avg_total_loss:.6f}, '
                      f'V={avg_voltage_loss:.6f}, M={avg_m_loss:.6f}, '
                      f'H={avg_h_loss:.6f}, N={avg_n_loss:.6f}')
    
    def plot_training_history(self):
        """绘制训练历史"""
        plt.figure(figsize=(15, 10))
        
        plt.subplot(2, 3, 1)
        plt.plot(self.train_losses)
        plt.title('Total Loss (HH Equations Only)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 2)
        plt.plot(self.voltage_losses)
        plt.title('Voltage Equation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 3)
        plt.plot(self.m_losses)
        plt.title('M Gate Equation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 4)
        plt.plot(self.h_losses)
        plt.title('H Gate Equation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.subplot(2, 3, 5)
        plt.plot(self.n_losses)
        plt.title('N Gate Equation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        
        plt.tight_layout()
        plt.show()

print("损失函数已更新为只使用HH微分方程约束，四个方程分别设置权重")


In [ ]:
# 更新物理约束类，添加分别计算四个HH方程损失的方法

class RNN_HH_PhysicsConstraints:
    """
    基于RNN的HH模型时序物理约束
    考虑时序依赖性的物理约束
    """
    
    def __init__(self, model):
        self.model = model
    
    def alpha_m(self, V):
        """钠通道m门控变量的激活速率"""
        return 0.1 * (V + 40.0) / (1.0 - torch.exp(-(V + 40.0) / 10.0))
    
    def beta_m(self, V):
        """钠通道m门控变量的失活速率"""
        return 4.0 * torch.exp(-(V + 65.0) / 18.0)
    
    def alpha_h(self, V):
        """钠通道h门控变量的激活速率"""
        return 0.07 * torch.exp(-(V + 65.0) / 20.0)
    
    def beta_h(self, V):
        """钠通道h门控变量的失活速率"""
        return 1.0 / (1.0 + torch.exp(-(V + 35.0) / 10.0))
    
    def alpha_n(self, V):
        """钾通道n门控变量的激活速率"""
        return 0.01 * (V + 55.0) / (1.0 - torch.exp(-(V + 55.0) / 10.0))
    
    def beta_n(self, V):
        """钾通道n门控变量的失活速率"""
        return 0.125 * torch.exp(-(V + 65.0) / 80.0)
    
    def compute_hh_equations_loss(self, sequence_input, predicted_outputs):
        """
        计算HH微分方程约束损失 - 分别返回四个方程的损失
        Args:
            sequence_input: [batch_size, seq_len, 5] -> [V(t-1), m(t-1), h(t-1), n(t-1), I(t)]
            predicted_outputs: [batch_size, seq_len, 4] -> [V(t), m(t), h(t), n(t)]
        Returns:
            voltage_loss, m_loss, h_loss, n_loss: 四个方程的损失
        """
        batch_size, seq_len, _ = predicted_outputs.shape
        
        # 分离输入和输出
        V_prev = sequence_input[:, :, 0:1]  # V(t-1)
        m_prev = sequence_input[:, :, 1:2]  # m(t-1)
        h_prev = sequence_input[:, :, 2:3]  # h(t-1)
        n_prev = sequence_input[:, :, 3:4]  # n(t-1)
        I = sequence_input[:, :, 4:5]       # I(t)
        
        V_curr = predicted_outputs[:, :, 0:1]  # V(t)
        m_curr = predicted_outputs[:, :, 1:2]  # m(t)
        h_curr = predicted_outputs[:, :, 2:3]  # h(t)
        n_curr = predicted_outputs[:, :, 3:4]  # n(t)
        
        # 计算时间导数（有限差分）
        dt = self.model.dt
        dV_dt = (V_curr - V_prev) / dt
        dm_dt = (m_curr - m_prev) / dt
        dh_dt = (h_curr - h_prev) / dt
        dn_dt = (n_curr - n_prev) / dt
        
        # HH方程约束
        # 1. 电压方程：C_m * dV/dt = I - g_Na*m^3*h*(V-E_Na) - g_K*n^4*(V-E_K) - g_L*(V-E_L)
        I_Na = self.model.g_Na * (m_curr**3) * h_curr * (V_curr - self.model.E_Na)
        I_K = self.model.g_K * (n_curr**4) * (V_curr - self.model.E_K)
        I_L = self.model.g_L * (V_curr - self.model.E_L)
        
        voltage_eq = self.model.C_m * dV_dt - (I - I_Na - I_K - I_L)
        voltage_loss = torch.mean(voltage_eq**2)
        
        # 2. m门控变量方程：dm/dt = alpha_m(V)*(1-m) - beta_m(V)*m
        m_eq = dm_dt - (self.alpha_m(V_prev) * (1 - m_prev) - self.beta_m(V_prev) * m_prev)
        m_loss = torch.mean(m_eq**2)
        
        # 3. h门控变量方程：dh/dt = alpha_h(V)*(1-h) - beta_h(V)*h
        h_eq = dh_dt - (self.alpha_h(V_prev) * (1 - h_prev) - self.beta_h(V_prev) * h_prev)
        h_loss = torch.mean(h_eq**2)
        
        # 4. n门控变量方程：dn/dt = alpha_n(V)*(1-n) - beta_n(V)*n
        n_eq = dn_dt - (self.alpha_n(V_prev) * (1 - n_prev) - self.beta_n(V_prev) * n_prev)
        n_loss = torch.mean(n_eq**2)
        
        return voltage_loss, m_loss, h_loss, n_loss

print("物理约束类已更新，添加了分别计算四个HH方程损失的方法")


In [ ]:
# 更新训练函数，使用新的纯HH微分方程约束损失函数

def train_rnn_hh_pinn(acquisition_dict_10khz, num_epochs=1000, batch_size=32, learning_rate=1e-3, sequence_length=200):
    """
    训练基于RNN的HH PINN模型 (使用10kHz降采样数据，纯HH微分方程约束)
    """
    print("开始RNN-based PINN训练 (使用10kHz降采样数据，纯HH微分方程约束)...")
    
    # 1. 数据处理
    print("1. 处理10kHz降采样电流钳数据为序列格式...")
    processor = RNN_PatchDataProcessor(acquisition_dict_10khz, sequence_length=sequence_length)
    processor.process_all_trials()
    
    # 可视化样本序列
    processor.plot_sample_sequences()
    
    # 创建训练数据集
    train_data, test_data = processor.create_training_dataset()
    train_loader, test_loader = create_rnn_data_loaders(train_data, test_data, batch_size)
    
    print(f"训练序列数: {len(train_data['sequences'])}")
    print(f"测试序列数: {len(test_data['sequences'])}")
    
    # 2. 初始化模型
    print("2. 初始化RNN PINN模型...")
    model = RNN_HH_PINN(hidden_size=128, num_layers=2, dropout=0.1)
    physics_constraints = RNN_HH_PhysicsConstraints(model)
    
    # 使用纯HH微分方程约束损失函数，四个方程分别设置权重
    loss_function = RNN_PINN_LossFunction(physics_constraints, 
                                        lambda_voltage=1.0, 
                                        lambda_m=1.0,
                                        lambda_h=1.0,
                                        lambda_n=1.0)
    trainer = RNN_PINN_Trainer(model, loss_function, learning_rate)
    
    # 打印模型参数
    print(f"模型参数数量: {sum(p.numel() for p in model.parameters())}")
    print(f"HH模型参数:")
    print(f"  g_Na: {model.g_Na.item():.2f}")
    print(f"  g_K: {model.g_K.item():.2f}")
    print(f"  g_L: {model.g_L.item():.2f}")
    print(f"  E_Na: {model.E_Na.item():.2f}")
    print(f"  E_K: {model.E_K.item():.2f}")
    print(f"  E_L: {model.E_L.item():.2f}")
    print(f"  C_m: {model.C_m.item():.2f}")
    print(f"  dt: {model.dt.item():.4f}")
    
    # 3. 训练模型
    print("3. 开始训练 (纯HH微分方程约束)...")
    trainer.train(train_loader, num_epochs=num_epochs, print_interval=100)
    
    # 4. 训练历史可视化
    print("4. 绘制训练历史...")
    trainer.plot_training_history()
    
    # 5. 模型评估
    print("5. 评估模型性能...")
    evaluate_rnn_model(model, test_loader, processor)
    
    return model, trainer, processor

def evaluate_rnn_model(model, test_loader, processor):
    """
    评估RNN模型性能 (纯HH微分方程约束)
    """
    model.eval()
    
    with torch.no_grad():
        test_losses = []
        all_predictions = []
        all_targets = []
        
        for sequence_input, true_outputs, initial_state in test_loader:
            predicted_outputs, hidden = model(sequence_input)
            
            # 计算测试损失 - 只使用HH微分方程约束
            physics_constraints = RNN_HH_PhysicsConstraints(model)
            loss_function = RNN_PINN_LossFunction(physics_constraints)
            total_loss, voltage_loss, m_loss, h_loss, n_loss = \
                loss_function.compute_total_loss(sequence_input, predicted_outputs, true_outputs, initial_state)
            
            test_losses.append(total_loss.item())
            
            # 收集预测结果
            all_predictions.extend(predicted_outputs.cpu().numpy())
            all_targets.extend(true_outputs.cpu().numpy())
        
        avg_test_loss = np.mean(test_losses)
        print(f"平均测试损失 (纯HH微分方程约束): {avg_test_loss:.6f}")
        
        # 计算相关系数
        predictions = np.array(all_predictions)
        targets = np.array(all_targets)
        
        # 计算电压预测的相关系数
        V_pred = predictions[:, :, 0].flatten()
        V_true = targets[:, :, 0].flatten()
        correlation = np.corrcoef(V_pred, V_true)[0, 1]
        print(f"电压预测与真实值的相关系数: {correlation:.4f}")
        
        # 绘制预测结果
        plot_rnn_predictions(predictions, targets, processor)

def plot_rnn_predictions(predictions, targets, processor, n_samples=5):
    """
    绘制RNN预测结果
    """
    plt.figure(figsize=(15, 12))
    
    # 随机选择样本进行可视化
    n_total = len(predictions)
    indices = np.random.choice(n_total, min(n_samples, n_total), replace=False)
    
    for i, idx in enumerate(indices):
        plt.subplot(n_samples, 1, i + 1)
        
        pred_sample = predictions[idx]
        target_sample = targets[idx]
        
        V_pred = pred_sample[:, 0]
        V_true = target_sample[:, 0]
        
        time_steps = range(len(V_pred))
        plt.plot(time_steps, V_true, 'b-', label='真实电压', alpha=0.8)
        plt.plot(time_steps, V_pred, 'r--', label='预测电压', alpha=0.8)
        
        plt.title(f'序列 {i+1} - 电压预测 (纯HH微分方程约束)')
        plt.xlabel('时间步')
        plt.ylabel('归一化电压')
        plt.legend()
        plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # 预测vs真实值散点图
    plt.figure(figsize=(10, 8))
    
    V_pred_all = predictions[:, :, 0].flatten()
    V_true_all = targets[:, :, 0].flatten()
    
    # 随机采样用于散点图
    n_scatter = min(5000, len(V_pred_all))
    scatter_indices = np.random.choice(len(V_pred_all), n_scatter, replace=False)
    
    plt.scatter(V_true_all[scatter_indices], V_pred_all[scatter_indices], 
                alpha=0.5, s=1, c='blue')
    plt.plot([V_true_all.min(), V_true_all.max()], 
             [V_true_all.min(), V_true_all.max()], 'r--', lw=2)
    plt.xlabel('真实电压')
    plt.ylabel('预测电压')
    plt.title('电压预测vs真实值 (纯HH微分方程约束)')
    plt.grid(True)
    plt.show()

def analyze_rnn_learned_parameters(model):
    """
    分析RNN学习到的HH模型参数
    """
    print("RNN学习到的HH模型参数 (纯HH微分方程约束训练):")
    print(f"钠电导 (g_Na): {model.g_Na.item():.2f} mS/cm²")
    print(f"钾电导 (g_K): {model.g_K.item():.2f} mS/cm²")
    print(f"漏电导 (g_L): {model.g_L.item():.2f} mS/cm²")
    print(f"钠平衡电位 (E_Na): {model.E_Na.item():.2f} mV")
    print(f"钾平衡电位 (E_K): {model.E_K.item():.2f} mV")
    print(f"漏平衡电位 (E_L): {model.E_L.item():.2f} mV")
    print(f"膜电容 (C_m): {model.C_m.item():.2f} μF/cm²")
    print(f"时间步长 (dt): {model.dt.item():.4f} s")
    
    # 与标准HH参数比较
    standard_params = {
        'g_Na': 120.0, 'g_K': 36.0, 'g_L': 0.3,
        'E_Na': 50.0, 'E_K': -77.0, 'E_L': -54.4, 'C_m': 1.0
    }
    
    print("\n与标准HH参数比较:")
    for param_name, standard_value in standard_params.items():
        learned_value = getattr(model, param_name).item()
        diff = abs(learned_value - standard_value)
        diff_pct = (diff / standard_value) * 100
        print(f"{param_name}: 学习值={learned_value:.2f}, 标准值={standard_value:.2f}, "
              f"差异={diff:.2f} ({diff_pct:.1f}%)")

print("训练函数已更新为使用纯HH微分方程约束损失函数")


In [ ]:
# 更新训练调用，使用纯HH微分方程约束

# 运行RNN PINN训练 (使用10kHz降采样数据，纯HH微分方程约束)
if 'acquisition_dict_10khz' in locals() and len(acquisition_dict_10khz) > 0:
    print("发现10kHz降采样电流钳数据，开始RNN-based PINN训练...")
    print("使用纯HH微分方程约束损失函数")
    print("四个方程分别设置权重: V=1.0, M=1.0, H=1.0, N=1.0")
    print("seq_length=200 (20ms)，适合RNN上下文长度")
    
    # 训练模型
    rnn_model, rnn_trainer, rnn_processor = train_rnn_hh_pinn(
        acquisition_dict_10khz, 
        num_epochs=500,  # 可以根据需要调整
        batch_size=16,   # RNN通常需要较小的batch size
        learning_rate=1e-3,
        sequence_length=200  # 20ms序列长度
    )
    
    # 分析学习到的参数
    analyze_rnn_learned_parameters(rnn_model)
    
else:
    print("未发现10kHz降采样数据，请先运行降采样处理部分...")
